# 05 Event study and matched placebo

Do crises produce more strong reactions than chance, and more than elections?

- **Matched placebo.** For every event, the pool of pairs (d, d+1) of consecutive normal days of its country,
  with d in the day class of day 0 and within `PLACEBO_RADIUS` calendar days of day 0 (`PLACEBO_RADIUS_WIDE`,
  once, if the pool has fewer than `PLACEBO_MIN_PAIRS` pairs). π(e) = share of pool pairs with a strong
  reaction: the probability of a strong reaction for that country, in that period, when nothing happens.
- **Excess rate** of a group: X = mean over its events of (Strong − π).
- **P1** (primary): X(crises) at p = `P_MAIN`, window day 0 and day +1, main labels, Israel included. One-sided
  p-value from the exact Poisson-binomial distribution of the number of strong crises with probabilities π.
- **P2** (primary): X(crises) − X(elections), same specification. Two-sided p-value from `N_GROUP_PERM`
  permutations of the group label within country.
- Confidence intervals: country-cluster bootstrap (`B_BOOT` replicates, percentile intervals at `CI_LEVEL`).
- The Holm correction across P1, P2 and P3 is computed once, in notebook 08 (P3 comes from 06).

Secondary analyses, labelled exploratory and reported in full: S1 elections vs placebo; S2 everything at
p = `P_SEC`; S3 day 0 only and day +1 only; S4 without Israel; S5 each robustness set; S6 COVID window dropped;
S7 planned cabinet changes vs placebo; S11 power of P1. The mean |SAR| by day around the events, with a
placebo band, is saved for the figure of notebook 08.

**Choices made in code, fixed before any result was looked at**:
- Every resampling procedure (bootstrap, permutation, placebo samples) uses its own numpy Generator seeded with
  `SEED`, so a result does not depend on the order of the cells.
- The bootstrap resamples the countries that have at least one event of the tested groups; a replicate in
  which a tested group is empty has no value and is counted (`n_boot_empty`).
- Permutation p-value: (1 + number of permuted statistics at least as large in absolute value) / (1 + N).
- Single-day windows (S3): Strong uses |AR| of that day only, and π the same day of the placebo pairs
  (d for day 0, d+1 for day +1).
- Placebo SAR paths are built like an event's path: the market model is fitted on the window placed around the
  placebo date d and applied to d−10 … d+10. A date whose path has a missing value is not drawn.
  The band of the mean |SAR| by day comes from `N_PLACEBO_SAMPLES` placebo samples of the same size.

In [1]:
import sys
from pathlib import Path

# On Nuvolos the kernel does not start in the notebook's folder: find code/ as 02_sample does.
roots = [Path.cwd(), Path("/files"), Path.home()]
code_dir = next(p for r in roots for p in [r, *r.glob("**/elections_project/code")] if (p / "helpers.py").exists())
sys.path.insert(0, str(code_dir))
print("Working in", code_dir)

import hashlib

import numpy as np
import pandas as pd

from helpers import (AR_WIN, B_BOOT, CI_LEVEL, COVID_WINDOW, N_GROUP_PERM, N_PLACEBO_SAMPLES, OUTCOME_DAYS,
                     P_MAIN, P_SEC, PATH_WIN, PLACEBO_MIN_PAIRS, PLACEBO_RADIUS, PLACEBO_RADIUS_WIDE,
                     POWER_RATES, SAMPLE_END, SEED, load_derived, load_prices, save_derived)
from analysis import (ANALYSIS_GROUPS, QUANTILES, apply_robustness_set, build_panels, cluster_bootstrap,
                      day_class, draw_placebo_paths, event_strong, group_permutation_test, load_robustness_sets,
                      market_model, normal_pairs, p1_power, percentile_interval, placebo_pool, placebo_rate,
                      poisson_binomial_upper, sar_paths)

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)

Working in /files/mywork/elections_project/code


## Inputs
Events and eligibility from 03, outcomes and eligibility from 04, normal days and thresholds from 04.
Every analysis sample comes from `apply_robustness_set(events, set, outcomes)`: events eligible in 03 and 04,
overlap rule applied among them.

In [2]:
events = load_derived("events", parse_dates=["source_date", "day0", "day1", "eligible_from"])
events["eligible"] = events["eligible"].astype(bool)
events["ineligible_reason"] = events["ineligible_reason"].fillna("")
outcomes = load_derived("event_outcomes")
outcomes["eligible"] = outcomes["eligible"].astype(bool)
normal = load_derived("normal_days", parse_dates=["date"])
normal["normal"] = normal["normal"].astype(bool)
thr = load_derived("thresholds")
political_dates = load_derived("political_dates", parse_dates=["date", "trading_day"])

q_main, q_sec = list(QUANTILES)
assert QUANTILES[q_main] == P_MAIN and QUANTILES[q_sec] == P_SEC
assert OUTCOME_DAYS == (0, 1), "placebo pairs are (d, d+1): the outcome window must be day 0 and day +1"

rob = load_robustness_sets()
SETS = ["main"] + list(dict.fromkeys(rob["set"]))
OUT_COLS = ["ar0", "ar1", "strong" + q_main[1:], "strong" + q_sec[1:]]


def analysis_sample(set_name):
    """Eligible events of one set after the overlap rule, with outcomes and the day classes of day 0 and +1."""
    t = apply_robustness_set(events, set_name, outcomes)
    t = t[t["eligible"] & ~t["overlap_drop"]].merge(outcomes[["event_id", "variant"] + OUT_COLS],
                                                    on=["event_id", "variant"], validate="one_to_one")
    t["class0"] = [day_class(c, [d])[0] for c, d in zip(t["country"], t["day0"])]
    t["class1"] = [day_class(c, [d])[0] for c, d in zip(t["country"], t["day1"])]
    return t.reset_index(drop=True)


samples = {s: analysis_sample(s) for s in SETS}

# Counts after eligibility in 03 and 04 and the overlap rule (printed by notebook 04)
expected = {"main": (76, 144, 43), "R_add": (83, 144, 38), "R_drop": (74, 144, 43), "R_vol": (70, 144, 43),
            "R_bundle": (78, 144, 41), "R_tv": (60, 144, 40), "R_split": (75, 144, 44)}
counts = {s: tuple(int((t["group"] == g).sum()) for g in ("crisis", "election", "planned"))
          for s, t in samples.items()}
assert counts == expected, counts
assert all(t[OUT_COLS].notna().all().all() for t in samples.values())
print(pd.DataFrame(counts, index=["crisis", "election", "planned"]).to_string())

          main  R_add  R_drop  R_vol  R_bundle  R_tv  R_split
crisis      76     83      74     70        78    60       75
election   144    144     144    144       144   144      144
planned     43     38      43     43        41    40       44


## Matched placebo pools and π
Pairs are built once from the normal days; each event row (main labels and alternative day 0) gets its pool
and its π for both thresholds and for the windows (0, +1), day 0 only and day +1 only.

In [3]:
pairs = normal_pairs(normal, thr)
WINDOWS = {"0,+1": OUTCOME_DAYS, "0": (OUTCOME_DAYS[0],), "+1": (OUTCOME_DAYS[1],)}
normal_keys = set(zip(normal.loc[normal["normal"], "country"], normal.loc[normal["normal"], "date"]))

all_rows = pd.concat(samples.values()).drop_duplicates(["event_id", "variant"])
pi_rows, pools = [], {}
for e in all_rows.itertuples(index=False):
    pool, radius = placebo_pool(pairs, e.country, e.day0, e.class0)
    assert len(pool) > 0, f"{e.event_id}: empty placebo pool; stop and decide what to do"
    pools[(e.event_id, e.variant)] = pool
    row = {"event_id": e.event_id, "variant": e.variant, "radius": radius, "pool_pairs": len(pool)}
    for name in QUANTILES:
        for w, days in WINDOWS.items():
            row[f"pi_{name}_{w}"] = placebo_rate(pool, name, days)
    pi_rows.append(row)
pi = pd.DataFrame(pi_rows)

# every placebo pair: two normal days of the event's country, d in day 0's class, inside the radius, d <= SAMPLE_END
for (eid, var), pool in pools.items():
    e = all_rows[(all_rows["event_id"] == eid) & (all_rows["variant"] == var)].iloc[0]
    r = pi.loc[(pi["event_id"] == eid) & (pi["variant"] == var), "radius"].iloc[0]
    assert (pool["country"] == e["country"]).all() and (pool["day_class"] == e["class0"]).all()
    assert ((pool["date"] - e["day0"]).abs().dt.days <= r).all() and (pool["date"] <= SAMPLE_END).all()
    assert all(k in normal_keys for k in zip(pool["country"], pool["date"]))
    assert all(k in normal_keys for k in zip(pool["country"], pool["date1"]))
pi_cols = [c for c in pi if c.startswith("pi_")]
assert pi[pi_cols].apply(lambda c: c.between(0, 1)).all().all()

wide = pi[pi["radius"] == PLACEBO_RADIUS_WIDE]
print(f"Event rows with a pool: {len(pi)}; radius widened to {PLACEBO_RADIUS_WIDE} days for {len(wide)} "
      f"(fewer than {PLACEBO_MIN_PAIRS} pairs within {PLACEBO_RADIUS} days)")
print("Smallest pools:", pi.nsmallest(5, "pool_pairs")[["event_id", "variant", "radius", "pool_pairs"]]
      .to_dict("records"))
main_pi = samples["main"].merge(pi, on=["event_id", "variant"])
print(f"Mean pi at p = {P_MAIN}, window (0, +1), main sample: {main_pi[f'pi_{q_main}_0,+1'].mean():.4f} "
      f"(1 - P_MAIN^2 = {1 - P_MAIN ** 2:.4f} if days were independent)")

Event rows with a pool: 267; radius widened to 1460 days for 0 (fewer than 100 pairs within 730 days)
Smallest pools: [{'event_id': 'ISR_1677', 'variant': 'main', 'radius': 730, 'pool_pairs': 124}, {'event_id': 'ISR_1158', 'variant': 'main', 'radius': 730, 'pool_pairs': 125}, {'event_id': 'ISR_1126', 'variant': 'main', 'radius': 730, 'pool_pairs': 131}, {'event_id': 'ISR_1132', 'variant': 'main', 'radius': 730, 'pool_pairs': 132}, {'event_id': 'ISR_1123', 'variant': 'main', 'radius': 730, 'pool_pairs': 137}]
Mean pi at p = 0.95, window (0, +1), main sample: 0.0921 (1 - P_MAIN^2 = 0.0975 if days were independent)


π should be close to, and slightly below, 1 − P_MAIN² = 0.0975: large |AR| days cluster, so a pair of
consecutive days has fewer "at least one" exceedances than two independent days (notebook 04 found pair shares
of 8.6–9.5% by country). A mean π far from that range would point to a bug in the pools.

**Consistency check.** The strong flags recomputed here from AR(0), AR(+1) and the thresholds must equal those
saved by 04.

In [4]:
for s, t in samples.items():
    for name in QUANTILES:
        assert (event_strong(t, thr, name, OUTCOME_DAYS) == t["strong" + name[1:]].astype(int)).all(), (s, name)
print("Strong flags recomputed here equal event_outcomes.csv.")

Strong flags recomputed here equal event_outcomes.csv.


## Tests
One function computes every row of the results table: the excess rate of one group against its placebo
(P1 for crises, S1 for elections, S7 for planned changes) with the exact one-sided Poisson-binomial p-value, or
the difference crises − elections (P2) with the within-country permutation p-value; every row gets a
country-cluster bootstrap interval.

In [5]:
def test_row(t, test, name, window, set_name, israel, covid):
    t = t.merge(pi, on=["event_id", "variant"], validate="one_to_one")
    if not israel:
        t = t[t["country"] != "ISR"]
    if covid == "dropped":
        lo, hi = (pd.Timestamp(x) for x in COVID_WINDOW)
        t = t[~(t["day0"].between(lo, hi) | t["day1"].between(lo, hi))]
    strong = event_strong(t, thr, name, WINDOWS[window])
    p_ev = t[f"pi_{name}_{window}"].to_numpy()
    d = strong - p_ev
    row = {"test": test, "set": set_name, "threshold": QUANTILES[name], "window": window, "israel": israel,
           "covid": covid}
    if test == "P2":
        sub = t["group"].isin(["crisis", "election"]).to_numpy()
        is_c = (t["group"] == "crisis").to_numpy()[sub]
        diff, p = group_permutation_test(t["country"].to_numpy()[sub], d[sub], is_c)
        reps, empty = cluster_bootstrap(t["country"].to_numpy()[sub], d[sub], is_c, ~is_c)
        row.update(group="crisis - election", N=int(sub.sum()), n_strong=np.nan, rate=np.nan, mean_pi=np.nan,
                   excess=diff, p_value=p)
    else:
        group = {"P1": "crisis", "S1": "election", "S7": "planned"}[test]
        g = (t["group"] == group).to_numpy()
        reps, empty = cluster_bootstrap(t["country"].to_numpy()[g], d[g], np.ones(g.sum(), dtype=bool))
        row.update(group=group, N=int(g.sum()), n_strong=int(strong[g].sum()), rate=strong[g].mean(),
                   mean_pi=p_ev[g].mean(), excess=d[g].mean(),
                   p_value=poisson_binomial_upper(strong[g].sum(), p_ev[g]))
    row["ci_low"], row["ci_high"] = percentile_interval(reps)
    row["n_boot_empty"] = empty
    row["primary"] = (test in ("P1", "P2") and set_name == "main" and name == q_main and window == "0,+1"
                      and israel and covid == "kept")
    return row


rows = []
# main labels: both thresholds (S2), the three windows (S3), with and without Israel (S4), all groups (S1, S7)
for name in QUANTILES:
    for window in WINDOWS:
        for israel in (True, False):
            for test in ("P1", "P2", "S1", "S7"):
                rows.append(test_row(samples["main"], test, name, window, "main", israel, "kept"))
# S5 robustness sets and S6 COVID window, at the primary specification
for s in SETS[1:]:
    for test in ("P1", "P2"):
        rows.append(test_row(samples[s], test, q_main, "0,+1", s, True, "kept"))
for test in ("P1", "P2"):
    rows.append(test_row(samples["main"], test, q_main, "0,+1", "main", True, "dropped"))

COLS = ["test", "group", "set", "threshold", "window", "israel", "covid", "N", "n_strong", "rate", "mean_pi",
        "excess", "ci_low", "ci_high", "p_value", "n_boot_empty", "primary"]
res = pd.DataFrame(rows)[COLS]
assert res["primary"].sum() == 2
assert ((res["mean_pi"].dropna() >= 0) & (res["mean_pi"].dropna() <= 1)).all()

### Primary tests (P1, P2), with the same tests without Israel next to them
No multiple-testing correction here: Holm across P1, P2 and P3 is applied in notebook 08.

In [6]:
show = ["test", "group", "israel", "N", "n_strong", "rate", "mean_pi", "excess", "ci_low", "ci_high", "p_value"]
prim = res[(res["set"] == "main") & (res["threshold"] == P_MAIN) & (res["window"] == "0,+1")
           & (res["covid"] == "kept") & res["test"].isin(["P1", "P2"])]
print(prim[show].round(4).to_string(index=False))

test             group  israel   N  n_strong   rate  mean_pi  excess  ci_low  ci_high  p_value
  P1            crisis    True  76      14.0 0.1842   0.0992  0.0850 -0.0045   0.1791   0.0138
  P2 crisis - election    True 220       NaN    NaN      NaN  0.0012 -0.1172   0.1305   0.9770
  P1            crisis   False  62      11.0 0.1774   0.0910  0.0864 -0.0189   0.1960   0.0201
  P2 crisis - election   False 195       NaN    NaN      NaN  0.0052 -0.1327   0.1477   0.9214


### Secondary analyses (exploratory)

In [7]:
print(res.drop(columns=["primary"]).round(4).to_string(index=False))

test             group      set  threshold window  israel   covid   N  n_strong   rate  mean_pi  excess  ci_low  ci_high  p_value  n_boot_empty
  P1            crisis     main       0.95   0,+1    True    kept  76      14.0 0.1842   0.0992  0.0850 -0.0045   0.1791   0.0138             0
  P2 crisis - election     main       0.95   0,+1    True    kept 220       NaN    NaN      NaN  0.0012 -0.1172   0.1305   0.9770             0
  S1          election     main       0.95   0,+1    True    kept 144      25.0 0.1736   0.0898  0.0838 -0.0013   0.1741   0.0007             0
  S7           planned     main       0.95   0,+1    True    kept  43       5.0 0.1163   0.0876  0.0287 -0.0503   0.1345   0.3213             0
  P1            crisis     main       0.95   0,+1   False    kept  62      11.0 0.1774   0.0910  0.0864 -0.0189   0.1960   0.0201             0
  P2 crisis - election     main       0.95   0,+1   False    kept 195       NaN    NaN      NaN  0.0052 -0.1327   0.1477   0.9214       

### Reproducibility
The primary rows are recomputed; with the same seed they must be identical.

In [8]:
again = pd.DataFrame([test_row(samples["main"], t, q_main, "0,+1", "main", True, "kept")
                      for t in ("P1", "P2")])[COLS]
assert again.equals(res[res["primary"]].reset_index(drop=True)), "P1/P2 differ between two runs"
print("P1 and P2 identical on a second run.")

P1 and P2 identical on a second run.


## S11 Power of P1
Critical count of the one-sided exact test at `ALPHA` under the Poisson-binomial null of the main crisis sample;
power = P(Binomial(N, rate) ≥ critical count) for each rate in `POWER_RATES`.

In [9]:
crises_main = samples["main"][samples["main"]["group"] == "crisis"].merge(pi, on=["event_id", "variant"])
power = p1_power(crises_main[f"pi_{q_main}_0,+1"].to_numpy(), POWER_RATES)
print(power.round(4).to_string(index=False))

 rate  n_events  critical_count   size  power
 0.15        76              13 0.0312 0.3498
 0.20        76              13 0.0312 0.7771
 0.25        76              13 0.0312 0.9622
 0.30        76              13 0.0312 0.9966


## Mean |SAR| by day around the events, with a placebo band
For crises, elections and planned changes (main labels): mean |SAR(t)| for t in `PATH_WIN`. Placebo band:
`N_PLACEBO_SAMPLES` samples of the same size, each event replaced by a date drawn from its own matched pool;
the band is the percentile interval at `CI_LEVEL` of the samples' mean |SAR(t)|. Placebo paths need the
market-model fits of every day, which are recomputed here with the functions of 04 and checked against the
event paths saved by 04.

In [10]:
panels = build_panels(load_prices(), political_dates)
fits = {c: market_model(p) for c, p in panels.items()}
rel = np.arange(PATH_WIN[0], PATH_WIN[1] + 1)
assert AR_WIN[0] <= PATH_WIN[0] and PATH_WIN[1] <= AR_WIN[1]

event_ar = load_derived("event_ar")
main = samples["main"]
ev_paths = (event_ar[event_ar["rel_day"].isin(rel)]
            .pivot(index=["event_id", "variant"], columns="rel_day", values="sar")
            .reindex(pd.MultiIndex.from_frame(main[["event_id", "variant"]]))[rel].to_numpy())

# the recomputed fits reproduce the event paths of 04
check = np.vstack([sar_paths(panels[c], fits[c], [panels[c].index.get_loc(d)], rel)
                   for c, d in zip(main["country"], main["day0"])])
assert np.allclose(check, ev_paths, equal_nan=True, rtol=0, atol=1e-9)

pool_paths = []
for e in main.itertuples(index=False):
    pool = pools[(e.event_id, e.variant)]
    pos = panels[e.country].index.get_indexer(pool["date"])
    assert (pos >= 0).all()
    pool_paths.append(sar_paths(panels[e.country], fits[e.country], pos, rel))

rng = np.random.default_rng(SEED)
placebo_means = {g: [] for g in ("crisis", "election", "planned")}
for _ in range(N_PLACEBO_SAMPLES):
    draw = np.abs(draw_placebo_paths(pool_paths, rng))
    for g in placebo_means:
        placebo_means[g].append(draw[(main["group"] == g).to_numpy()].mean(axis=0))

sar_rows = []
for g in placebo_means:
    sel = (main["group"] == g).to_numpy()
    band = np.array(placebo_means[g])
    low, high = (np.percentile(band, q, axis=0) for q in (100 * (1 - CI_LEVEL) / 2, 100 * (1 + CI_LEVEL) / 2))
    sar_rows.append(pd.DataFrame({
        "group": g, "rel_day": rel, "n_events": np.isfinite(ev_paths[sel]).sum(axis=0),
        "mean_abs_sar": np.nanmean(np.abs(ev_paths[sel]), axis=0),
        "placebo_median": np.median(band, axis=0), "placebo_low": low, "placebo_high": high}))
mean_abs_sar = pd.concat(sar_rows, ignore_index=True)
print(mean_abs_sar.round(3).to_string(index=False))

   group  rel_day  n_events  mean_abs_sar  placebo_median  placebo_low  placebo_high
  crisis      -10        76         0.806           0.772        0.620         0.922
  crisis       -9        76         0.846           0.765        0.605         0.965
  crisis       -8        76         0.790           0.789        0.631         0.999
  crisis       -7        76         0.864           0.768        0.601         0.954
  crisis       -6        76         0.787           0.764        0.597         1.029
  crisis       -5        76         0.774           0.761        0.601         0.983
  crisis       -4        76         0.842           0.788        0.639         1.009
  crisis       -3        76         0.833           0.768        0.626         0.949
  crisis       -2        76         0.868           0.763        0.630         0.916
  crisis       -1        76         0.691           0.759        0.619         0.963
  crisis        0        76         1.046           0.808        

## Outputs
- `res_event_study.csv`: one row per test and variant (P1, P2 primary; S1–S7 secondary).
- `res_mean_abs_sar_by_day.csv`: mean |SAR| by day and group, with the placebo band.
- `res_power.csv`: S11.

In [11]:
for nm, df in [("res_event_study", res), ("res_mean_abs_sar_by_day", mean_abs_sar), ("res_power", power)]:
    print(save_derived(df, nm), len(df), "rows")
print("res_event_study hash:", hashlib.sha256(res.to_csv(index=False).encode()).hexdigest())

/files/mywork/elections_project/data/derived/res_event_study.csv 62 rows
/files/mywork/elections_project/data/derived/res_mean_abs_sar_by_day.csv 63 rows
/files/mywork/elections_project/data/derived/res_power.csv 4 rows
res_event_study hash: 10855d8176f19d236086d8b732c4fe4efd202672b8147ce1a774690e13fe9e5e
